In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

import torch
import json
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.cuda.amp import autocast

from notebooks.local.utils import get_paths, create_folders, safe_cosine_similarity

PATHS    = get_paths()
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
THRESHOLDS = [0.05, 0.10, 0.20]

SIM_DIR = os.path.join(PATHS['step4_ens'], 'sim_maps')
os.makedirs(SIM_DIR, exist_ok=True)
create_folders(PATHS)
print("Device:", DEVICE, "  Sim maps dir:", SIM_DIR)


In [ ]:
import os
import shutil, tarfile
from notebooks.local.utils import get_paths, create_folders, download_file

PATHS    = get_paths()
create_folders(PATHS)

spair_check = os.path.join(PATHS['spair71k'], 'JPEGImages')
if not os.path.exists(spair_check):
    print('SPair-71k not found. Downloading (~2 GB) ...')
    tar_path = os.path.join(PATHS['data'], 'SPair-71k.tar.gz')
    download_file(
        'http://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz',
        tar_path, desc='SPair-71k',
    )
    print('Extracting ...')
    with tarfile.open(tar_path, 'r:gz') as t:
        t.extractall(PATHS['spair71k'])
    extracted_sub = os.path.join(PATHS['spair71k'], 'SPair-71k')
    if os.path.isdir(extracted_sub):
        for item in os.listdir(extracted_sub):
            shutil.move(os.path.join(extracted_sub, item),
                        os.path.join(PATHS['spair71k'], item))
        os.rmdir(extracted_sub)
    os.remove(tar_path)
    print('SPair-71k ready.')
else:
    print('SPair-71k already present.')

if not os.path.exists(PATHS['dinov2_w']):
    print('Downloading DINOv2 ViT-B/14 weights (~330 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth',
        PATHS['dinov2_w'], desc='DINOv2',
    )
else:
    print('DINOv2 weights present.')

if not os.path.exists(PATHS['sam_w']):
    print('Downloading SAM ViT-B weights (~370 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        PATHS['sam_w'], desc='SAM',
    )
else:
    print('SAM weights present.')

if not os.path.exists(PATHS['dinov3_w']):
    print('WARNING: DINOv3 weights not found at', PATHS['dinov3_w'])
    print('  Place dinov3_vitb16_pretrain.pth in weights/ (obtain from project maintainer).')
else:
    print('DINOv3 weights present.')


In [ ]:
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as vit_base_v2
from src.models.dinov3.dinov3.models.vision_transformer import vit_base as vit_base_v3
from src.models.segment_anything.segment_anything import sam_model_registry
from src.datasets.spair_dataset import SPairDataset
from src.features.extractor import (
    extract_dense_features, extract_dense_features_SAM,
    pixel_to_patch_coord,
)
from src.metrics.pck import compute_pck_spair71k
from src.matching.strategies import find_best_match_argmax, find_best_match_window_softargmax
from experiments.evaluate_ensemble import LearnedEnsembleWeights


pair_ann = os.path.join(PATHS['spair71k'], 'PairAnnotation')
layout   = os.path.join(PATHS['spair71k'], 'Layout')
images   = os.path.join(PATHS['spair71k'], 'JPEGImages')
train_ds = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'trn')
val_ds   = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'val')
test_ds  = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'test')
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")


def extract_and_save_sims(model, dataset, backbone, img_size, patch_size, split,
                           sim_dir, device, is_sam=False):
    """Extract per-keypoint similarity maps and save to disk."""
    out_dir = os.path.join(sim_dir, backbone, split)
    os.makedirs(out_dir, exist_ok=True)

    model.eval()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            out_path = os.path.join(out_dir, f"pair_{idx:05d}.pt")
            if os.path.exists(out_path):
                continue

            src_t = sample['src_img'].unsqueeze(0).to(device)
            tgt_t = sample['trg_img'].unsqueeze(0).to(device)
            if USE_FP16 and not is_sam:
                src_t, tgt_t = src_t.half(), tgt_t.half()

            src_orig = (sample['src_imsize'][2], sample['src_imsize'][1])

            if is_sam:
                src_feat = extract_dense_features_SAM(model, src_t, image_size=img_size)
                tgt_feat = extract_dense_features_SAM(model, tgt_t, image_size=img_size)
            else:
                src_feat = extract_dense_features(model, src_t)
                tgt_feat = extract_dense_features(model, tgt_t)

            _, H, W, D = tgt_feat.shape
            tgt_flat = tgt_feat.reshape(H * W, D)

            src_kps = sample['src_kps'].numpy()
            sims_list = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(
                    src_kps[i, 0], src_kps[i, 1], src_orig, patch_size, img_size)
                sf = src_feat[0, py, px, :]
                sims = safe_cosine_similarity(sf.float(), tgt_flat.float())
                sims_list.append(sims.cpu())

            torch.save({'sims': sims_list, 'H': H, 'W': W,
                        'src_kps': sample['src_kps'],
                        'trg_kps': sample['trg_kps'],
                        'trg_bbox': sample['trg_bbox'],
                        'src_imsize': sample['src_imsize'],
                        'trg_imsize': sample['trg_imsize'],
                        'category': sample['category'],
                        'kps_ids': sample['kps_ids']},
                       out_path)

            if (idx + 1) % 500 == 0:
                print(f"  {split} {idx+1}/{len(dataset)}")

    print(f"Sim maps saved for {backbone}/{split}: {out_dir}")


## Step 1 — Extract Similarity Maps: DINOv2

In [ ]:
backbone = 'dinov2'
img_size, patch_size = 518, 14

model = vit_base_v2(img_size=(518,518), patch_size=14,
                    num_register_tokens=0, block_chunks=0, init_values=1.0)
ft_path = PATHS['dinov2_ft']
if os.path.exists(ft_path):
    ckpt = torch.load(ft_path, map_location=DEVICE, weights_only=True)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state, strict=True)
    print("Loaded fine-tuned DINOv2")
else:
    ckpt = torch.load(PATHS['dinov2_w'], map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt, strict=True)
    print("Loaded pretrained DINOv2")

model = model.to(DEVICE)
if USE_FP16:
    model = model.half()

for split, ds in [('trn', train_ds), ('val', val_ds), ('test', test_ds)]:
    extract_and_save_sims(model, ds, backbone, img_size, patch_size, split,
                          SIM_DIR, DEVICE, is_sam=False)

del model; torch.cuda.empty_cache()
print("DINOv2 sim extraction complete.")


## Step 2 — Extract Similarity Maps: DINOv3

In [ ]:
backbone = 'dinov3'
img_size, patch_size = 512, 16

model = vit_base_v3(img_size=512, patch_size=16, n_storage_tokens=4, mask_k_bias=True, layerscale_init=1.0e-05, norm_layer="layernormbf16")
ft_path = PATHS['dinov3_ft']
if os.path.exists(ft_path):
    ckpt = torch.load(ft_path, map_location=DEVICE, weights_only=True)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state, strict=True)
    print("Loaded fine-tuned DINOv3")
else:
    ckpt = torch.load(PATHS['dinov3_w'], map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt, strict=True)
    print("Loaded pretrained DINOv3")

model = model.to(DEVICE)
if USE_FP16:
    model = model.half()

for split, ds in [('trn', train_ds), ('val', val_ds), ('test', test_ds)]:
    extract_and_save_sims(model, ds, backbone, img_size, patch_size, split,
                          SIM_DIR, DEVICE, is_sam=False)

del model; torch.cuda.empty_cache()
print("DINOv3 sim extraction complete.")


## Step 3 — Extract Similarity Maps: SAM

In [ ]:
backbone = 'sam'
img_size, patch_size = 512, 16

model = sam_model_registry['vit_b'](checkpoint=PATHS['sam_w'])
ft_path = PATHS['sam_ft']
if os.path.exists(ft_path):
    ckpt = torch.load(ft_path, map_location=DEVICE, weights_only=True)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state, strict=False)
    print("Loaded fine-tuned SAM")

model = model.to(DEVICE)

for split, ds in [('trn', train_ds), ('val', val_ds), ('test', test_ds)]:
    extract_and_save_sims(model, ds, backbone, img_size, patch_size, split,
                          SIM_DIR, DEVICE, is_sam=True)

del model; torch.cuda.empty_cache()
print("SAM sim extraction complete.")


## Step 4 — Train Learned Ensemble Weights

In [ ]:
from src.matching.strategies import find_best_match_window_softargmax
from src.features.extractor import patch_to_pixel_coord

K           = 5
TEMPERATURE = 0.10
LR_ENS      = 1e-2
ENS_EPOCHS  = 50

learned_w_path = PATHS['learned_w']

if os.path.exists(learned_w_path):
    print("Learned weights already saved:", learned_w_path)
    with open(learned_w_path) as f:
        learned_weights = json.load(f)
    print("Weights:", learned_weights)
else:
    weights_model = LearnedEnsembleWeights(n_models=3).to(DEVICE)
    optimizer_ens = torch.optim.Adam(weights_model.parameters(), lr=LR_ENS)

    backbones = ['dinov2', 'dinov3', 'sam']
    n_train = len(train_ds)
    IMG_SIZES = {'dinov2': 518, 'dinov3': 512, 'sam': 512}
    PATCH_SIZES = {'dinov2': 14, 'dinov3': 16, 'sam': 16}

    best_val_loss = float('inf')
    for epoch in range(ENS_EPOCHS):
        weights_model.train()
        total_loss, n = 0.0, 0
        for idx in range(n_train):
            sims_list = []
            meta = None
            for bb in backbones:
                p = os.path.join(SIM_DIR, bb, 'trn', f"pair_{idx:05d}.pt")
                if not os.path.exists(p):
                    break
                d = torch.load(p, map_location=DEVICE, weights_only=False)
                sims_list.append([s.to(DEVICE) for s in d['sims']])
                if meta is None:
                    meta = d
            if len(sims_list) < 3:
                continue

            w = weights_model()  # [3]
            n_kps = len(sims_list[0])
            trg_kps = meta['trg_kps'].numpy()
            trg_bbox = meta['trg_bbox']
            H, W = meta['H'], meta['W']

            loss_kps = []
            for ki in range(n_kps):
                fused = sum(w[bi] * sims_list[bi][ki] for bi in range(3))
                fused_log = F.log_softmax(fused * TEMPERATURE, dim=0)
                tgt_x_gt = int(trg_kps[ki, 0])
                tgt_y_gt = int(trg_kps[ki, 1])
                tgt_orig = (meta['trg_imsize'][2], meta['trg_imsize'][1])
                px, py = pixel_to_patch_coord(tgt_x_gt, tgt_y_gt, tgt_orig, 14, 518)
                gt_idx = py * W + px
                gt_idx = max(0, min(gt_idx, H * W - 1))
                loss_kps.append(-fused_log[gt_idx])

            if loss_kps:
                loss = torch.stack(loss_kps).mean()
                optimizer_ens.zero_grad()
                loss.backward()
                optimizer_ens.step()
                total_loss += loss.item(); n += 1

        avg_loss = total_loss / max(n, 1)
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{ENS_EPOCHS}: avg_loss={avg_loss:.4f}  weights={[round(x,3) for x in weights_model().tolist()]}")

    final_weights = weights_model().detach().cpu().tolist()
    learned_weights = {'dinov2': final_weights[0], 'dinov3': final_weights[1], 'sam': final_weights[2]}
    with open(learned_w_path, 'w') as f:
        json.dump(learned_weights, f, indent=2)
    print("Learned ensemble weights:", learned_weights)


## Step 5 — Evaluate Ensemble on SPair-71k Test

In [ ]:
out_dir = os.path.join(PATHS['step4_ens'], 'eval')
os.makedirs(out_dir, exist_ok=True)
stats_path = os.path.join(out_dir, 'overall_stats.json')

if os.path.exists(stats_path):
    print("Ensemble eval — already done.")
    with open(stats_path) as f:
        ens_stats = json.load(f)
    print(ens_stats)
else:
    with open(learned_w_path) as f:
        lw = json.load(f)
    w = torch.tensor([lw['dinov2'], lw['dinov3'], lw['sam']], device=DEVICE)

    backbones = ['dinov2', 'dinov3', 'sam']
    IMG_SIZES = {'dinov2': 518, 'dinov3': 512, 'sam': 512}
    PATCH_SIZES = {'dinov2': 14, 'dinov3': 16, 'sam': 16}

    per_img, all_kp = [], []
    for idx in range(len(test_ds)):
        sims_list = []
        meta = None
        for bb in backbones:
            p = os.path.join(SIM_DIR, bb, 'test', f"pair_{idx:05d}.pt")
            if not os.path.exists(p):
                break
            d = torch.load(p, map_location=DEVICE, weights_only=False)
            sims_list.append([s.to(DEVICE) for s in d['sims']])
            if meta is None:
                meta = d
        if len(sims_list) < 3:
            continue

        H, W = meta['H'], meta['W']
        trg_kps = meta['trg_kps'].numpy()
        trg_bbox = meta['trg_bbox']
        tgt_orig = (meta['trg_imsize'][2], meta['trg_imsize'][1])
        pred = []

        for ki in range(len(sims_list[0])):
            fused = sum(w[bi] * sims_list[bi][ki] for bi in range(3))
            mx, my = find_best_match_window_softargmax(fused, W, H, K=K, temperature=TEMPERATURE)
            rx, ry = patch_to_pixel_coord(mx, my, tgt_orig, 14, 518)
            pred.append([rx, ry])

        image_pcks = {}
        for thr in THRESHOLDS:
            pck, correct_mask, dists = compute_pck_spair71k(pred, trg_kps.tolist(), trg_bbox, thr)
            image_pcks[thr] = pck
            for kid, p_, g, d_, c in zip(meta['kps_ids'], pred, trg_kps.tolist(), dists, correct_mask):
                all_kp.append({'image_idx': idx, 'category': meta['category'],
                               'keypoint_id': kid, 'correct_at_threshold': c, 'threshold': thr})
        per_img.append({'category': meta['category'], 'pck_scores': image_pcks})

        if (idx + 1) % 500 == 0:
            print(f"  {idx+1}/{len(test_ds)}")

    ens_stats = {}
    for thr in THRESHOLDS:
        vals = [m['pck_scores'][thr] for m in per_img]
        ens_stats[f"pck@{thr:.2f}"] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
        print(f"Ensemble PCK@{thr:.2f}: {np.mean(vals):.2f}%")

    with open(stats_path, 'w') as f:
        json.dump(ens_stats, f, indent=2)
    print("Ensemble eval done.")


## Summary

In [ ]:
rows = []
configs = [
    ('DINOv2', os.path.join(PATHS['step1'], 'dinov2_argmax', 'overall_stats.json')),
    ('DINOv3', os.path.join(PATHS['step1'], 'dinov3_argmax', 'overall_stats.json')),
    ('SAM',    os.path.join(PATHS['step1'], 'sam_argmax',    'overall_stats.json')),
    ('Ensemble (learned)', os.path.join(PATHS['step4_ens'], 'eval', 'overall_stats.json')),
]
for label, sp in configs:
    if os.path.exists(sp):
        with open(sp) as f:
            s = json.load(f)
        rows.append({
            'Model': label,
            'PCK@0.05': round(s.get('pck@0.05',{}).get('mean', float('nan')), 2),
            'PCK@0.10': round(s.get('pck@0.10',{}).get('mean', float('nan')), 2),
            'PCK@0.20': round(s.get('pck@0.20',{}).get('mean', float('nan')), 2),
        })
    else:
        rows.append({'Model': label, 'PCK@0.05': '-', 'PCK@0.10': '-', 'PCK@0.20': '-'})

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
